In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pymargins import GComputation

rng = np.random.default_rng(42)
n = 2000
df = pd.DataFrame({
    "age": rng.integers(20, 75, n),
    "female": rng.binomial(1, 0.52, n),
    "treated": rng.binomial(1, 0.40, n),
})
lp = -1.5 + 0.04 * df["age"] - 0.3 * df["female"] + 0.8 * df["treated"]
df["y"] = rng.binomial(1, 1 / (1 + np.exp(-lp)))

fit = smf.glm("y ~ age + female + treated", data=df,
              family=sm.families.Binomial()).fit(disp=0)

In [2]:
est = GComputation(fit, at="overall", scale="response", method="auto", n_sim=4000)
print(est.plan.describe())

Plan b105456@1
  method: delta (declared: auto)
  resolution reason: auto: posture κ=0.120 ≤ 0.3
  scale: response
  at: overall
  ci: wald
  level: 0.95
  B: 1000
  n_sim: 4000
  seed: None
  data fingerprint: e76dfc713b3c80c8...


In [3]:
est = GComputation(fit, at="overall", scale="response", method="delta")
res = est.predict(atexog={"age": [25, 45, 65]})
print(res.kappa)

[0.01918631 0.02054411 0.04510454]


In [4]:
print(est.plan.describe())

Plan 79e47fe@1
  method: delta (declared: delta)
  resolution reason: user-specified
  scale: response
  at: overall
  ci: wald
  level: 0.95
  B: 1000
  n_sim: 4000
  seed: None
  data fingerprint: e76dfc713b3c80c8...


In [5]:
# Always use delta, regardless of κ
est_delta = GComputation(fit, at="overall", scale="response", method="delta")

# Always use simulation / Krinsky–Robb
est_sim = GComputation(fit, at="overall", scale="response", method="simulation", n_sim=4000)